# 08 · Common index: definition, data and validation protocol
Change of objective approved by the user: a common scale, without fixing Aqua
as ground truth. Notebook 07 stays as a comparison baseline.
This notebook does not fit the index or open the final 2024–2025 test.
An index is not considered correct just because it makes the curves match.

## 1. What we want to measure
Relative detectable thermal activity per zone and week, with the sensor effect
and observation opportunities modelled explicitly. Not the real number of
fires, ignition probability or burned area.
Let A(g,t) be the estimated common activity and B(g) its mean over a fixed development
base period. We define I(g,t)=100*A(g,t)/B(g), only if B(g)>0 and there is enough
support. I=100 is the historical average of that zone; I=200 means twice
the estimated activity relative to that base, not twice the fires.
Do not use the minimum/maximum of the whole history or add a hidden epsilon if B=0.
Indices with local bases compare relative deviations, not absolute
magnitudes between regions. A and its defined unit/support are also kept.

## 2. Candidate model, not a mathematical guarantee
Proposal to test: a joint observation model, with Poisson counts
as baseline and negative binomial if the observed dispersion justifies it.
An initial form is log(mu_sgt)=log(E_sgt)+eta_gt+alpha_s+delta_s,season.
E represents valid observation area-time on a common geographic support;
it is not simply the number of points or the raw number of 375 m pixels versus 1 km.
eta is the common latent activity and alpha/delta are response differences.
Detection also depends on intensity/size: test extensions only
if they improve validation and keep identifiability and stability.
Identification constraints are required (for example sensor effects
summing to zero) as well as connectivity of the overlap periods. Activity, coverage and
sensitivity cannot be estimated freely from positives alone.
Fixing the scale does not turn the latent activity into an observed physical truth.

In [1]:
from pathlib import Path
import os,json
import pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name in {'notebooks','validation'}:
    ROOT=ROOT.parent
ARTIFACTS=Path(os.environ.get('FIRMS_ARTIFACTS_DIR',str(ROOT/'artifacts')))
PREP=ARTIFACTS/'02_preparacion'
OUT=ARTIFACTS/'09_indice_comun'
OUT.mkdir(parents=True,exist_ok=True)
inventory=pd.read_csv(PREP/'inventario.csv')
display(inventory[['source_member','processing','rows','start','end']])

,source_member,processing,rows,start,end
0,fire_archive_M-C61_848898.csv,SP,362159,2000-11-01 04:16:00+00:00,2026-08-31 20:44:00+00:00
1,fire_nrt_M-C61_848898.csv,NRT,4264,2026-09-01 02:04:00+00:00,2026-11-13 02:02:00+00:00
2,fire_archive_J1V-C2_848899.csv,SP,641945,2018-04-01 05:13:00+00:00,2026-07-31 18:13:00+00:00
3,fire_nrt_J1V-C2_848899.csv,NRT,44689,2026-08-01 05:18:00+00:00,2026-11-13 18:47:00+00:00
4,fire_archive_J2V-C2_848900.csv,SP,243041,2023-03-01 06:21:00+00:00,2026-07-31 18:59:00+00:00
5,fire_nrt_J2V-C2_848900.csv,NRT,41185,2026-08-01 06:01:00+00:00,2026-11-14 06:33:00+00:00
6,fire_archive_SV-C2_848901.csv,SP,1058423,2012-01-20 06:33:00+00:00,2026-07-31 19:34:00+00:00
7,fire_nrt_SV-C2_848901.csv,NRT,43565,2026-08-01 06:37:00+00:00,2026-11-01 18:51:00+00:00


## 3. Requirements audit
The fields required in this table are an internal contract of the future exposure
table; NASA is not assumed to use these same column names.
The presence of the schema is examined; its semantics or quality are not validated yet.
Full fire/observation masks can distinguish cloud, missing,
non-fire and fire. The detection CSV files do not provide all those states.
NASA source: https://appliedsciences.nasa.gov/sites/default/files/2025-05/Part%203%20FIRMS%202025%20Q%26A%20Session.docx.pdf
VIIRS algorithm: https://viirsland.gsfc.nasa.gov/PDF/VIIRS_activefire_375m_ATBD.pdf

In [2]:
required_exposure={'valid_observed_area_km2','observation_time_utc','platform','cell_id','fire_mask_class'}
checks=[]
for _,row in inventory.iterrows():
    path=PREP/row.output_file
    columns=set(pd.read_csv(path,nrows=0).columns)
    missing=sorted(required_exposure-columns)
    checks.append({'file':row.source_member,'exposure_schema_present':not missing,
                   'missing_internal_fields':','.join(missing)})
checks=pd.DataFrame(checks)
display(checks)
exposure_ready=bool(checks.exposure_schema_present.all())
assert exposure_ready is False, 'Si se agregan fuentes de exposición, revisar contrato y semántica antes de entrenar.'
checks.to_csv(OUT/'auditoria_exposicion.csv',index=False)
print('Fuentes positivas disponibles; exposición observacional completa todavía no incorporada.')

,file,exposure_schema_present,missing_internal_fields
0,fire_archive_M-C61_848898.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
1,fire_nrt_M-C61_848898.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
2,fire_archive_J1V-C2_848899.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
3,fire_nrt_J1V-C2_848899.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
4,fire_archive_J2V-C2_848900.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
5,fire_nrt_J2V-C2_848900.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
6,fire_archive_SV-C2_848901.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."
7,fire_nrt_SV-C2_848901.csv,False,"cell_id,fire_mask_class,observation_time_utc,v..."


Fuentes positivas disponibles; exposición observacional completa todavía no incorporada.


## 4. Validation pipeline
Complementary layers are run; none replaces the others.
Fire precision/recall requires representative references and verifiable
negative cases. The four positive INDECI reports are not enough.

In [3]:
pipeline=pd.DataFrame([
 {'stage':'Datos','checks':'Esquema, UTC, coordenadas, versiones, duplicados y trazabilidad','reuse':'01_firms_preparation','status':'base implementada'},
 {'stage':'Observación','checks':'Pasos, máscara, nubes, área válida, ceros observados frente a faltantes','reuse':'fuentes nuevas de máscaras/exposición','status':'pendiente de datos'},
 {'stage':'Espacial','checks':'R-tree más distancia geodésica; celdas coherentes; bloques espaciales reservados y separación entre bloques','reuse':'02_spatiotemporal_organization y 05_curimana_comparison','status':'asignación implementada; validación espacial externa pendiente'},
 {'stage':'Temporal','checks':'Backtesting por bloques; normalización y selección ajustadas dentro de cada fold; prueba final intacta','reuse':'03_diagnostico y 07_baseline','status':'baseline disponible; protocolo común pendiente'},
 {'stage':'Entre plataformas','checks':'Aqua/Terra/SNPP/NOAA20/NOAA21 separados; dejar una plataforma fuera; concordancia y discontinuidad al cambiar','reuse':'períodos de solapamiento y datos ya descargados','status':'pendiente'},
 {'stage':'Evidencia externa','checks':'Fecha del hecho, georreferencia e incertidumbre; no atribuir candidatos automáticamente','reuse':'04_auditoria_json y validation/05_curimana_comparison','status':'caso exploratorio implementado'},
 {'stage':'Revisión humana','checks':'Coherencia geográfica, fecha de imagen, notas y exportación','reuse':'validation/06_reusable_google_maps','status':'integración pendiente de API key real'},
 {'stage':'Robustez','checks':'Región/estación/día-noche/resolución; sesgo residual por sensor; incertidumbre y faltantes','reuse':'07 como control, no objetivo final','status':'pendiente para índice común'}
])
display(pipeline)
pipeline.to_csv(OUT/'pipeline_validacion.csv',index=False)

,stage,checks,reuse,status
0,Datos,"Esquema, UTC, coordenadas, versiones, duplicad...",01_preparacion_firms,base implementada
1,Observación,"Pasos, máscara, nubes, área válida, ceros obse...",fuentes nuevas de máscaras/exposición,pendiente de datos
2,Espacial,R-tree más distancia geodésica; celdas coheren...,02_organizacion_espacio_temporal y 05_contrast...,asignación implementada; validación espacial e...
3,Temporal,Backtesting por bloques; normalización y selec...,03_diagnostico y 07_baseline,baseline disponible; protocolo común pendiente
4,Entre plataformas,Aqua/Terra/SNPP/NOAA20/NOAA21 separados; dejar...,períodos de solapamiento y datos ya descargados,pendiente
5,Evidencia externa,"Fecha del hecho, georreferencia e incertidumbr...",04_auditoria_json y validacion/05_contraste_cu...,caso exploratorio implementado
6,Revisión humana,"Coherencia geográfica, fecha de imagen, notas ...",validacion/06_google_maps_reutilizable,integración pendiente de API key real
7,Robustez,Región/estación/día-noche/resolución; sesgo re...,"07 como control, no objetivo final",pendiente para índice común


## 5. Evaluation that must be fixed before the common fit
- Hold out 2024–2025 for the final test; that block has not been evaluated.
- Review development/calibration for NOAA-21: it starts in 2023, so its
  effect cannot be learned using only 2013–2021.
- Define spatial blocks and buffers before looking at their errors. Do not mix
  neighbouring cells of the same episode between fitting and independent evaluation.
- Estimate exposure, normalization, sensor effects and hyperparameters only
  inside the fitting block of each temporal/spatial fold.
- Evaluate held-out observations: MAE, bias, deviance appropriate to the model,
  interval coverage and stability by segments; do not measure error against
  the latent index itself as if it were ground truth.
- Transition test: remove a sensor/platform and check continuity
  using the remaining ones, without forcing real environmental changes to disappear.
- Freeze the method before opening the final test. If it is changed afterwards,
  that test is no longer independent and must be declared exploratory.

In [4]:
contract={
 'objective':'common_relative_detectable_thermal_activity_index',
 'reference_sensor':None,'physical_truth_claim':False,'index_fitted':False,
 'baseline_notebook_07_retained':True,'base_definition':'100 times estimated common activity / fixed development mean for each region',
 'base_zero_policy':'not estimable; no hidden epsilon',
 'coverage_adjusted_model_ready':exposure_ready,
 'needs':['observational_exposure_and_fire_masks','NOAA20_NOAA21_overlap_analysis','identified_joint_observation_model','predeclared_spatial_and_temporal_validation'],
 'test_2024_2025_evaluated':False,
 'four_positive_reports_establish_global_precision_recall':False,
 'google_maps_confirms_2019_fire_automatically':False,
 'next_step':'audit availability of observation-mask products and select one pilot region/time window',
}
(OUT/'contrato_indice_comun.json').write_text(json.dumps(contract,ensure_ascii=False,indent=2),encoding='utf-8')
print('Contrato registrado. No se entrenó un índice común con exposición desconocida.')

Contrato registrado. No se entrenó un índice común con exposición desconocida.
